In [39]:
import pandas as pd
from  collections import defaultdict
import math

In [50]:
def sub_nan(val):
    if isinstance(val, float) and math.isnan(val):
        return None
    return val

class CandidateInfo:
    def __init__(self, df_row):
        self.comp_name = df_row['comp_name']
        self.candidate_type = df_row['candidate_type']
        self.candidate_id = df_row['candidate_id']

        self.pdb_id_b = df_row['pdb_id_b']
        self.ab_chain_ids_b = df_row['ab_chain_ids_b'].split(':')
        self.ag_chain_ids_b = df_row['ag_chain_ids_b'].split(':')

        self.ab_pdb_id_u = df_row['ab_pdb_id_u']
        self.ab_chain_ids_u = df_row['ab_chain_ids_u'].split(':')
        self.ag_pdb_id_u = df_row['ag_pdb_id_u']
        self.ag_chain_ids_u = df_row['ag_chain_ids_u'].split(':')
        
        self.small_mols_msg = sub_nan(df_row['small_molecules_message'])

In [51]:
df = pd.read_csv('db_info.csv', dtype=str)

complexes = set()
candidate_infos = defaultdict(list)

for i in range(len(df)):
    candidate_info = CandidateInfo(df.iloc[i])

    if candidate_info.candidate_type != 'U:U':
        continue

    if candidate_info.comp_name not in complexes:
        complexes.add(candidate_info.comp_name)
        
    candidate_infos[candidate_info.comp_name].append(candidate_info)

In [52]:
deleted = set()

df = pd.read_csv('duplicates.csv', dtype=str)

duplicates = defaultdict(list)

for i in range(len(df)):
    duplicates[df.iloc[i]['comp_name']].append(df.iloc[i]['duplicate_name'])
    
for comp_name, duplicates in duplicates.items():
    if comp_name in deleted:
        continue
        
    for x in duplicates:
        deleted.add(x)
        candidate_infos[comp_name] += candidate_infos[x]
        complexes.remove(x)

In [53]:
len(complexes)

101

In [58]:
len(list(filter(lambda x: any(map(lambda y: y.small_mols_msg is None or y.small_mols_msg == 'small molecules with 5 < n_atoms <= 15 detected', candidate_infos[x])), complexes)))

85